In [2]:
import sys
import os
from dotenv import load_dotenv
sys.path.append('..')
env_file = os.path.join('..', '.env')
load_dotenv(env_file)
dataset_path = os.environ['DATASET_PATH']
test_path = os.environ['TEST_PATH']
test_coco_path = os.environ['TEST_COCO_PATH']
coco_path = os.environ['COCO_PATH']
cache_pdf = os.environ['CASH_PDF_PATH']

In [5]:
from rows2regionsGLAM.utils.pdf_manager import PDFManager
from rows2regionsGLAM.utils.loger import Loger
from rows2regionsGLAM.utils.row_manager import RowManager
from rows2regionsGLAM.utils.ploter import Ploter
from rows2regionsGLAM.tokenizers import RowGLAMTokenizer
from rows2regionsGLAM.utils.coco_manager import COCOManager
from rows2regionsGLAM.datasetloaders.base_line_dataset import GLAMDataset

In [6]:
loger = Loger('log_20260127.txt')
pdf_manager = PDFManager(conf={"loger": loger, "pdf_reader": "PDFMiner"})
row_manager = RowManager(conf={"loger": loger, "add_image": True})
coco_manager = COCOManager(conf={"loger": loger, "coco_path": coco_path})
ploter = Ploter(conf={"loger": loger})
tokenizer = RowGLAMTokenizer()
loger(tokenizer.get_name())

In [7]:
from rows2regionsGLAM.metrics import GridMetric
from torchmetrics.detection.mean_ap import MeanAveragePrecision

/Users/macbookair/program/python/PageR/env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
from pager.page_model.sub_models.dtype import ImageSegment


def get_true_edges(token, rows, region_segs, region_categories):
    def is_one_region(row_seg_1, row_seg_2, region_segs):
        for i, reg in enumerate(region_segs):
            if reg.is_intersection(row_seg_1):
                if reg.is_intersection(row_seg_2):
                    return 1
                else:
                    return 0
        return 0

    def get_category(seg, region_segs, region_categories):
        for r, c in zip(region_segs, region_categories):
            if seg.is_intersection(r):
                return c
        return None

    def get_mini_seg(r):
        img_seg = ImageSegment(dict_2p=r)
        if img_seg.height < 5:
            return img_seg
        delta =  int(img_seg.height/5)
        img_seg.y_bottom_right = img_seg.y_bottom_right - delta
        img_seg.y_top_left = img_seg.y_top_left + delta
        return img_seg
    row_segments = [get_mini_seg(row['segment']) for row in rows]
    A = token['inds']
    true_edges = [is_one_region(row_segments[i], row_segments[j], region_segs) for i, j in zip(A[0], A[1])]
    true_nodes = [get_category(row_seg, region_segs, region_categories) for row_seg in row_segments]
    return true_edges, true_nodes

def pdf2torch_dict(path_pdf, coco_dict_file):
        try:
            pdf_json = pdf_manager.get_json_from_pdf(path_pdf, num_page=0)
            row_json = row_manager.get_row_json_from_pdf_json(pdf_json)
        except:
            print(path_pdf, '\n')
            return {}
        torch_dict = tokenizer(row_json)
        coef_w, coef_h = 1, 1
        coco_dict_file['regions'] = [r for r in coco_dict_file['regions'] if r['segment']['height'] > 0]
        reg_segments = [ImageSegment(dict_p_size={
            "x_top_left": int(r['segment']['x_top_left']*coef_w),
            "y_top_left": int(r['segment']['y_top_left']*coef_h),
            "width": int(r['segment']['width']*coef_w),
            "height": int(r['segment']['height']*coef_h)}) for r in coco_dict_file['regions']]
        reg_categories = [r['category_id'] for r in coco_dict_file['regions']]
        true_edges, true_nodes = get_true_edges(torch_dict, row_json, reg_segments, reg_categories)
        
        del torch_dict['sp_A']
        torch_dict['true_edges'] = true_edges
        torch_dict['true_nodes'] = true_nodes

        return torch_dict
         
        print(f"{(i+1)/N*100:4.2f} %", end='\r')

test_dataset = GLAMDataset(
    {
    "loger": loger,
    "pdf_dir": test_path,
    "coco_file": test_coco_path,
    "count_class": 6,
    "default_index": 0,
    "cache_dir": cache_pdf,
    "pdf2torch_dict": pdf2torch_dict
    }
)

In [28]:
from pager.page_model.sub_models import BaseConverter, RegionModel, RowsModel
from pager.page_model.sub_models.dtype import ImageSegment, Region, Graph
from pager import MergeExtractor


CLASSES = {1: 'text', 2: 'header', 3: 'text', 4: 'table', 5: 'figure', 0: 'other'}

class Rows2Regions(BaseConverter):
    def __init__(self, conf):
        self.rows2regionsGLAM_tokenizer = conf['tokenizer']# manager_model.get_model("rowGLAM-tokenizer")
        self.is_merge_extract =  conf['is_merge_extract']
        self.merge_extract = MergeExtractor()

        
    def convert(self, input_model: RowsModel, output_model: RegionModel):
        page_json = input_model.to_dict()
        true_edges = input_model.true_edges
        node_classes = input_model.node_classes
        region_list = self.get_region(page_json['rows'],true_edges,node_classes)
        output_model.from_dict({"regions": region_list})

        if self.is_merge_extract:
            self.merge_extract.extract(output_model)
        # сортировка после создания региона
        # sorter = RegionSorterCutXYExtractor()
        # sorter.extract(output_model)

    def get_region(self, rows_json, true_edges,node_classes):
        graph_dict_torch = self.rows2regionsGLAM_tokenizer(rows_json)
        result = dict()
        result['deleted_edges'] = true_edges < 0.5
        result['node_classes'] = node_classes
        
        graph = graph_dict_torch['inds']
        deleted_edges = result['deleted_edges']
        node_classes = result['node_classes']
        regions = self.regions_from_graph(rows_json, graph, deleted_edges, node_classes)
        return regions
    

    def regions_from_graph(self, rows_json, graph, deleted_edges, node_classes):
        graph_ = Graph()
        regions = []
        
        for row_json in rows_json:
            segment = ImageSegment(dict_2p=row_json['segment'])
            xc, yc = segment.get_center()
            graph_.add_node(xc, yc)

        for node_i, node_j, ind in zip(graph[0], graph[1], deleted_edges):
            if not ind:
                graph_.add_edge(node_i+1, node_j+1)

        for reg in graph_.get_related_graphs():
            indexes = [node.index-1 for node in reg.get_nodes()]
            row_classes  = np.array([node_classes[i].detach().numpy() for i in indexes])
            lable = CLASSES[np.argmax(row_classes.mean(axis=0))]
            regions.append({'rows': [rows_json[i] for i in indexes], 'label': lable})
        return regions

In [29]:
rows_model = RowsModel()
region_model = RegionModel()
rows2regions = Rows2Regions({
    'tokenizer': RowGLAMTokenizer(),
    'is_merge_extract': True
})

In [31]:
import numpy as np
from pager import ImageSegment

def get_bbox(segment, resize = None, delta_w = 0, delta_h = 0):
    coef_w, coef_h = 1, 1
    if resize:
        coef_w, coef_h = resize
    if not "height" in segment:
        segment['width'] = segment['x_bottom_right']-segment['x_top_left']
        segment['height']= segment['y_bottom_right']-segment['y_top_left']
    return [
        int(coef_w*segment['x_top_left']-delta_w),
        int(coef_h*segment['y_top_left']-delta_h),
        int(coef_w*segment['width']+delta_w),
        int(coef_h*segment['height']+delta_h)
    ]

def clean_rows(rows, bboxes_true):
    def is_row_in_region(row, segs_regions):
        for r in segs_regions:
            if row.is_intersection(r):
                return True
        return False
    def is_good_block(block):
        h = block['segment']['y_bottom_right']-block['segment']['y_top_left']
        w = block['segment']['x_bottom_right']-block['segment']['x_top_left']
        return h > 3 and w > 3
        
    old_rows = [row for row in rows if is_good_block(row)]
    segs_row = [ImageSegment(dict_2p=row['segment']) for row in old_rows]
    new_rows = []
    seg_bboxes_true = [ImageSegment(dict_p_size=bbox) for bbox in bboxes_true]
    for i, row in enumerate(segs_row):
        if is_row_in_region(row, seg_bboxes_true):
            new_rows.append(old_rows[i])
    rows = new_rows

def clean_bboxes_true(bboxes_true):
    return [bbox_true for bbox_true in bboxes_true if bbox_true['height'] > 3 and bbox_true['width'] > 3]





target = []
preds = []
word_grids = []
row_grids = []
paths = []
N = len(test_dataset)
for i, d in enumerate(test_dataset):
    name_file = test_dataset.pdf_names[i]
    true_regions = test_dataset.coco_ann[name_file]['regions']
    bboxes_true = clean_bboxes_true([reg['segment'] for reg in true_regions])
    
    pdf_json = pdf_manager.get_json_from_pdf(os.path.join(test_path, name_file))
    row_json = row_manager.get_row_json_from_pdf_json(pdf_json)
    clean_rows(row_json, bboxes_true)
    
    rows_model.from_dict({"rows": row_json})
    rows_model.true_edges = d['true_edges']
    rows_model.node_classes = d['true_nodes']
    
    rows2regions.convert(rows_model, region_model)
    bboxes_pred = [reg['segment'] for reg in region_model.to_dict()['regions'] if reg['label'] != 'other']

    word_grids.append([get_bbox(word['segment']) for row in row_json for word in row['words']])
    row_grids.append([get_bbox(row['segment']) for row in row_json])
    target.append([get_bbox(seg) for seg in bboxes_true])
    preds.append([get_bbox(seg) for seg in bboxes_pred])

    
    print(f"{(i+1)/N*100:4.2f} %", end='\r')

35.00 %

Cannot set gray non-stroke color because /'P1' is an invalid float value
Cannot set gray non-stroke color because /'P2' is an invalid float value
Cannot set gray non-stroke color because /'P3' is an invalid float value
Cannot set gray non-stroke color because /'P4' is an invalid float value


45.00 %

Cannot set gray non-stroke color because /'P5' is an invalid float value
Cannot set gray non-stroke color because /'P6' is an invalid float value
Cannot set gray non-stroke color because /'P7' is an invalid float value
Cannot set gray non-stroke color because /'P8' is an invalid float value


53.00 %

Cannot set gray non-stroke color because /'P1' is an invalid float value


54.50 %

Cannot set gray non-stroke color because /'P2' is an invalid float value
Cannot set gray non-stroke color because /'P3' is an invalid float value
Cannot set gray non-stroke color because /'P3' is an invalid float value
Cannot set gray non-stroke color because /'P4' is an invalid float value
Cannot set gray non-stroke color because /'P4' is an invalid float value
Cannot set gray non-stroke color because /'P5' is an invalid float value
Cannot set gray non-stroke color because /'P5' is an invalid float value


100.00 %

In [32]:
grid_metric = GridMetric()

i = 0
N = len(preds)
for bboxes_pred, bboxes_true, grid_row, grid_word in zip(preds, target, row_grids, word_grids):
    i+=1
    try:
        grid_metric.add_pair(bboxes_pred, bboxes_true, grid_row, grid_word)
        print(f"{(i)/N*100:4.2f} %", end='\r')
    except:
        pass
grid_metric.update() 

100.00 %

In [33]:
import torch
map_metric = MeanAveragePrecision(box_format="xywh")

get_category = lambda an: 1

map_metric.update([dict(     
                    boxes=torch.tensor(bboxes_pred) ,
                    scores=torch.tensor([1.0 for an in bboxes_pred]),
                    labels=torch.tensor([get_category(an) for an in bboxes_pred]),
                    ) for bboxes_pred in preds], 
                  [dict(     
                        boxes=torch.tensor(bboxes_true) ,
                        labels=torch.tensor([get_category(an) for an in bboxes_true]),
                        ) for bboxes_true in target])  
rez = map_metric.compute()
map_metric_rez = f"mAP@IoU[0.50:0.95]   :{rez['map']:.8f}"

In [34]:
print(map_metric_rez)
print(grid_metric)
loger("Test Result")
loger(map_metric_rez)
loger(grid_metric.__str__())

mAP@IoU[0.50:0.95]   :0.60089475
threshold_05--------------------
precision_row       :0.9777
recall_row          :0.9581
f1_row              :0.9678
precision_word      :0.9365
recall_word         :0.9184
f1_word             :0.9274
threshold_95--------------------
precision_row       :0.9062
recall_row          :0.8892
f1_row              :0.8976
precision_word      :0.8706
recall_word         :0.8549
f1_word             :0.8627

